# Phase 4 — Hyperparameter Optimization, Calibration, Operating Points & Error Analysis
**Project:** AI Agent Conversation Quality Scorer · **Date:** 2026-06-11 · **Session 4 of 7**

**Where Phase 3 left us.** The champion `eng_xgboost` (grounding-overlap ⊕ 12 engineered claim-relation
features) hit **0.9967 raw / 0.9808 length-matched** macro-F1 — *near-saturated*. Phase 3's verdict was
explicit: *"the bottleneck is the features, not the model"* (an overlap-free XGB tied the full one), and the
handoff said Phase 4 should stop chasing raw score and instead probe **calibration, operating point for the
entity-reuse minority, and a standing form-ambiguous control.**

**Today's questions.**
1. Does systematic hyperparameter search (Optuna, grouped CV) actually move a saturated, feature-bound model?
2. Is XGBoost even the right family, or do LightGBM / CatBoost / a calibrated linear head win once each is tuned?
3. Is the champion's *probability* trustworthy (calibration / ECE / Brier), and does isotonic vs Platt help?
4. What operating point should ship — and how does each trade recall on the **entity-reuse minority** (the
   hard hallucinations) against precision?
5. *Where exactly does it fail?* Confusion structure, real FP/FN examples, systematic error buckets.
6. Is more data the answer, or is the model saturated? (learning curve)

**Research that shaped today (Phase 4a).**
- *Optuna / XGBoost tuning* (Random Realizations; Optuna docs): TPE Bayesian search, ~30+ trials usually
  suffice, scope the space to budget. → I cap each study by wall-clock and tune 9 XGB knobs.
- *Calibrating boosted trees* (Niculescu-Mizil & Caruana, "Obtaining Calibrated Probabilities from Boosting";
  FastML): isotonic most consistently improves GBMs but needs data; Platt assumes a sigmoid distortion.
  ECE = weighted mean |acc − conf| per bin. → I compare uncalibrated / Platt / isotonic with a held-out
  calibration split (grouped, so the calibrator never sees data the base model trained on).
- *Threshold-moving* (MachineLearningMastery; PR-curve guidance): 0.5 is rarely optimal; pick the operating
  point from explicit FN:FP costs and report the PR curve, not just ROC.
- *"The Mirage of Hallucination Detection"* (Findings-EMNLP 2025) & *"The Gray Zone of Faithfulness"* (2025):
  trained hallucination classifiers are distribution-shift sensitive and faithfulness is genuinely ambiguous.
  → motivates keeping the **form-ambiguous** slice as a permanent honesty control.

**Primary metric:** macro-F1 (frozen since Phase 1). Every model reported on the **raw** test split **and** the
**length-matched control** (HaluEval-QA leaks a 4.85× answer-length shortcut). Bar to clear: **0.9244** matched
(Phase-2 lexical overlap); current champion **0.9808** matched.

In [1]:
import json, os, re, time, warnings, difflib, bisect
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "4"
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from scipy.stats import ks_2samp
import sklearn, scipy
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_root():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand/"data"/"raw"/"qa_data.json").exists(): return cand
    raise RuntimeError("repo root not found")
ROOT = find_root(); RESULTS = ROOT/"results"; CACHE = RESULTS/"phase4_cache"; MODELS = ROOT/"models"
CACHE.mkdir(parents=True, exist_ok=True)
print("sklearn", sklearn.__version__, "| scipy", scipy.__version__)
print("root:", ROOT)

sklearn 1.8.0 | scipy 1.17.0
root: /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/AI-Agent-Conversation-Quality-Scorer


## 1 · Rebuild the frozen dataset, split, length-matched control, and the 12 engineered features
Identical pipeline to Phases 1–3 (same `qid` split, same nearest-length matched control, same feature code) so
every number here is directly comparable to the prior leaderboards. The engineered feature matrix is cached to
`results/phase4_cache/eng_F.npy` keyed by the stable 0..19999 row positions.

In [2]:
raw_path = ROOT/"data"/"raw"/"qa_data.json"
records = [json.loads(l) for l in raw_path.read_text().splitlines() if l.strip()]
long = []
for qid, r in enumerate(records):
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["right_answer"], "label": 0})
    long.append({"qid": qid, "knowledge": r["knowledge"], "question": r["question"], "answer": r["hallucinated_answer"], "label": 1})
df = pd.DataFrame(long).reset_index(drop=True)
df["ans_chars"] = df.answer.str.len()

STOP = set("a an the of to in on at for and or is was were are be been by with as that this it from".split())
def toks(s):     return [t for t in re.findall(r"[a-z0-9]+", str(s).lower()) if t not in STOP]
def toks_all(s): return re.findall(r"[a-z0-9]+", str(s).lower())
def grounding_overlap(ans, know):
    a = set(toks(ans)); k = set(toks(know)); return (len(a & k)/len(a)) if a else 0.0
df["ground_overlap"] = [grounding_overlap(a,k) for a,k in zip(df.answer, df.knowledge)]

split = json.load(open(RESULTS/"phase1_split_qids.json"))
train_qids, test_qids = set(split["train_qids"]), set(split["test_qids"])
train = df[df.qid.isin(train_qids)]; test = df[df.qid.isin(test_qids)]

def nearest_length_match(frame, caliper=8):
    g0 = frame[frame.label==0].sort_values("ans_chars"); g1 = frame[frame.label==1].sort_values("ans_chars")
    chars0 = g0.ans_chars.tolist(); idx0 = g0.index.tolist(); keep0, keep1 = [], []
    for c1, i1 in zip(g1.ans_chars.tolist(), g1.index.tolist()):
        if not chars0: break
        p = bisect.bisect_left(chars0, c1); best = None
        for j in (p-1, p, p+1):
            if 0 <= j < len(chars0):
                d = abs(chars0[j]-c1)
                if best is None or d < best[0]: best = (d, j)
        d, j = best
        if d <= caliper:
            keep1.append(i1); keep0.append(idx0[j]); del chars0[j]; del idx0[j]
    return frame.loc[keep0 + keep1]
matched = nearest_length_match(test, caliper=8)
def ks(fr): return ks_2samp(fr[fr.label==0].ans_chars, fr[fr.label==1].ans_chars).statistic
print(f"train {len(train)} | test {len(test)} | matched n={len(matched)} | "
      f"answer-length KS raw={ks(test):.3f} -> matched={ks(matched):.3f}")

train 16000 | test 4000 | matched n=572 | answer-length KS raw=0.874 -> matched=0.122


In [3]:
# --- 12 engineered claim-relation features (identical to Phase 3) ---
NUM_RE  = re.compile(r"\d+(?:\.\d+)?"); YEAR_RE = re.compile(r"\b(?:1[0-9]{3}|20[0-9]{2})\b")
NEG = {"not","no","never","none","cannot","without","neither","nor","n't","dont","didnt","doesnt","isnt","wasnt","werent","arent","wont","cant"}
def split_sents(txt):
    s = re.split(r"(?<=[.!?])\s+", str(txt).strip()); return [x for x in s if x.strip()] or [str(txt)]
docfreq = Counter()
for s in pd.concat([train.knowledge, train.answer]): docfreq.update(set(toks(s)))
NDOC = 2*len(train)
def idf(t): return np.log((NDOC+1)/(docfreq.get(t,0)+1))
def feat_row(ans, q, know):
    a_lc = str(ans).lower().strip().rstrip("."); k_lc = str(know).lower()
    aset = set(toks(ans)); kset = set(toks(know)); at_all = toks_all(ans); kt_all = toks_all(know)
    is_substr = float(a_lc in k_lc) if a_lc else 0.0
    lcs_char = (difflib.SequenceMatcher(None, a_lc, k_lc, autojunk=False).find_longest_match(0,len(a_lc),0,len(k_lc)).size/len(a_lc)) if a_lc else 0.0
    lcs_tok = (difflib.SequenceMatcher(None, at_all, kt_all, autojunk=False).find_longest_match(0,len(at_all),0,len(kt_all)).size/len(at_all)) if at_all else 0.0
    anum = set(NUM_RE.findall(str(ans))); knum = set(NUM_RE.findall(str(know)))
    num_frac = (len(anum & knum)/len(anum)) if anum else 1.0
    n_miss = len(anum - knum)
    ayr = set(YEAR_RE.findall(str(ans))); year_mismatch = len(ayr - set(YEAR_RE.findall(str(know))))
    sent_max = 0.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s)); o = len(aset & ss)/len(aset) if ss else 0.0
            if o > sent_max: sent_max = o
    whole = (len(aset & kset)/len(aset)) if aset else 0.0
    spread = whole - sent_max
    novel = aset - set(toks(q)); novel_ov = (len(novel & kset)/len(novel)) if novel else 1.0
    num = sum(idf(t) for t in aset if t in kset); den = sum(idf(t) for t in aset)
    idf_ov = (num/den) if den else 0.0
    ans_neg = float(any(w in NEG for w in at_all) or "n't" in str(ans).lower())
    best_s, best = "", -1.0
    if aset:
        for s in split_sents(know):
            ss = set(toks(s)); o = len(aset & ss)/len(aset) if ss else 0.0
            if o > best: best, best_s = o, s
    sent_neg = float(any(w in NEG for w in toks_all(best_s)) or "n't" in best_s.lower())
    neg_mismatch = float(ans_neg != sent_neg)
    return (is_substr, lcs_char, lcs_tok, num_frac, n_miss, year_mismatch, sent_max, spread, novel_ov, idf_ov, ans_neg, neg_mismatch)

ENG = ["is_substr","lcs_char_ratio","lcs_token_ratio","num_frac_in_know","n_num_missing","year_mismatch",
       "sent_overlap_max","overlap_spread","novel_overlap","idf_overlap","ans_has_neg","neg_mismatch"]
fcache = CACHE/"eng_F.npy"
if fcache.exists():
    F = np.load(fcache); print("loaded cached engineered features", F.shape)
else:
    t0=time.time(); F = np.array([feat_row(a,q,k) for a,q,k in zip(df.answer, df.question, df.knowledge)], dtype=np.float32)
    np.save(fcache, F); print(f"engineered {F.shape} in {time.time()-t0:.1f}s")
for j,name in enumerate(ENG): df[name] = F[:,j]
ALL = ["ground_overlap"] + ENG
Xtr, ytr = df.loc[train.index, ALL].values, train.label.values
Xte, yte = df.loc[test.index, ALL].values, test.label.values
Xmt, ymt = df.loc[matched.index, ALL].values, matched.label.values
grp_tr = train.qid.values
print("feature matrix:", Xtr.shape, "| classes balanced:", np.bincount(ytr))

loaded cached engineered features (20000, 12)
feature matrix: (16000, 13) | classes balanced: [8000 8000]


## 2 · The honest baseline: the *default* champion under grouped cross-validation
Before tuning anything, pin the default `eng_xgboost` (Phase 3's exact config: 500 trees, depth 4, lr 0.05).
The CV here uses **StratifiedGroupKFold by `qid`** — an item's grounded & hallucinated answers never straddle a
fold, so the CV score is leakage-free and is what Optuna will optimize. We report CV macro-F1 plus the held-out
raw and length-matched scores (should reproduce 0.9967 / 0.9808).

In [4]:
from xgboost import XGBClassifier
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(sgkf.split(Xtr, ytr, groups=grp_tr))
print("fold sizes (val):", [len(v) for _,v in FOLDS], "| group-disjoint:",
      all(set(grp_tr[tr]).isdisjoint(set(grp_tr[va])) for tr,va in FOLDS))

def cv_macro_f1(make_model, X=Xtr, y=ytr, folds=FOLDS):
    sc=[]
    for tr_i, va_i in folds:
        m = make_model(); m.fit(X[tr_i], y[tr_i])
        p = np.asarray(m.predict(X[va_i])).ravel().astype(int)
        sc.append(f1_score(y[va_i], p, average="macro"))
    return float(np.mean(sc)), float(np.std(sc))

def metrics(y, pred, score):
    return {"accuracy": round(accuracy_score(y,pred),4), "macro_f1": round(f1_score(y,pred,average='macro'),4),
            "balanced_acc": round(balanced_accuracy_score(y,pred),4),
            "precision_hallu": round(precision_score(y,pred,pos_label=1,zero_division=0),4),
            "recall_hallu": round(recall_score(y,pred,pos_label=1,zero_division=0),4),
            "roc_auc": round(roc_auc_score(y,score),4) if len(set(y))>1 else None}

DEFAULT = dict(n_estimators=500, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
               tree_method="hist", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=4)
t0=time.time(); def_cv, def_cv_sd = cv_macro_f1(lambda: XGBClassifier(**DEFAULT))
def_model = XGBClassifier(**DEFAULT).fit(Xtr, ytr)
def_raw = f1_score(yte, def_model.predict(Xte), average="macro")
def_mat = f1_score(ymt, def_model.predict(Xmt), average="macro")
print(f"DEFAULT eng_xgboost  CV={def_cv:.4f}±{def_cv_sd:.4f}  raw={def_raw:.4f}  matched={def_mat:.4f}  ({time.time()-t0:.0f}s)")
print("(Phase-3 reported raw=0.9967 matched=0.9808 -> reproduced)")

fold sizes (val): [4000, 4000, 4000, 4000] | group-disjoint: True


DEFAULT eng_xgboost  CV=0.9951±0.0016  raw=0.9967  matched=0.9808  (77s)
(Phase-3 reported raw=0.9967 matched=0.9808 -> reproduced)


## 3 · Experiment 4.1 — Optuna search over the XGBoost champion (9 knobs, grouped-CV objective)
**Hypothesis (from Phase 3):** the model is feature-bound, so Bayesian search will buy *very little*. TPE
sampler, objective = mean grouped-CV macro-F1, capped at ~4 min wall-clock. Knobs: trees, depth, learning rate,
subsample, colsample, min_child_weight, gamma, reg_alpha, reg_lambda.

In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 900, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        tree_method="hist", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=4)
    m, _ = cv_macro_f1(lambda: XGBClassifier(**p))
    return m
xgb_storage = "sqlite:///" + str(CACHE/"optuna_xgb.db")
study = optuna.create_study(study_name="xgb_p4", storage=xgb_storage, direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), load_if_exists=True)
if len(study.trials) == 0:
    study.enqueue_trial({k:v for k,v in DEFAULT.items() if k in
        ("n_estimators","max_depth","learning_rate","subsample","colsample_bytree")})  # seed with the default
if len(study.trials) < 40:
    t0=time.time(); study.optimize(obj_xgb, n_trials=120, timeout=300, show_progress_bar=False)
    print(f"{len(study.trials)} trials in {time.time()-t0:.0f}s | best CV macro-F1 = {study.best_value:.4f}")
else:
    print(f"resumed cached XGB study: {len(study.trials)} trials | best CV macro-F1 = {study.best_value:.4f}")
best_xgb_params = {**study.best_params, "tree_method":"hist", "eval_metric":"logloss",
                   "random_state":RANDOM_STATE, "n_jobs":4}
print("best params:", json.dumps(study.best_params, indent=0))

11 trials in 310s | best CV macro-F1 = 0.9949
best params: {
"n_estimators": 500,
"max_depth": 4,
"learning_rate": 0.05,
"subsample": 0.9,
"colsample_bytree": 0.9,
"min_child_weight": 4,
"gamma": 4.75357153204958,
"reg_alpha": 0.03872090295370417,
"reg_lambda": 0.0024430162614261434
}


In [6]:
# refit tuned champion on full train; held-out raw + matched; delta vs default
tuned = XGBClassifier(**best_xgb_params).fit(Xtr, ytr)
tuned_raw = f1_score(yte, tuned.predict(Xte), average="macro")
tuned_mat = f1_score(ymt, tuned.predict(Xmt), average="macro")
print(f"TUNED   eng_xgboost  CV={study.best_value:.4f}  raw={tuned_raw:.4f}  matched={tuned_mat:.4f}")
print(f"DEFAULT eng_xgboost  CV={def_cv:.4f}  raw={def_raw:.4f}  matched={def_mat:.4f}")
print(f"Δ from {len(study.trials)} Optuna trials:  CV {study.best_value-def_cv:+.4f}  "
      f"raw {tuned_raw-def_raw:+.4f}  matched {tuned_mat-def_mat:+.4f}")
tdf = study.trials_dataframe().sort_values("value", ascending=False).head(10)
cols = [c for c in tdf.columns if c=="value" or c.startswith("params_")]
trials_tbl = tdf[cols].rename(columns=lambda c: c.replace("params_","")).reset_index(drop=True)
print("\nTop-10 trials by CV macro-F1:"); print(trials_tbl.round(4).to_string(index=False))
trials_tbl.to_csv(RESULTS/"phase4_tuning_trials.csv", index=False)

TUNED   eng_xgboost  CV=0.9949  raw=0.9967  matched=0.9808
DEFAULT eng_xgboost  CV=0.9951  raw=0.9967  matched=0.9808
Δ from 11 Optuna trials:  CV -0.0002  raw +0.0000  matched +0.0000

Top-10 trials by CV macro-F1:
 value  colsample_bytree  gamma  learning_rate  max_depth  min_child_weight  n_estimators  reg_alpha  reg_lambda  subsample
0.9949            0.9000 4.7536         0.0500          4                 4           500     0.0387      0.0024     0.9000
0.9949            0.8099 1.4561         0.0187          4                 5           350     0.0032      0.0000     0.7217
0.9949            0.9234 0.4884         0.2521          3                 4           300     0.0144      0.0001     0.9863
0.9949            0.9579 4.6094         0.1396         10                 6           300     0.0000      0.0000     0.9758
0.9949            0.9315 1.4047         0.0375          5                 4           200     0.0008      0.0000     0.7085
0.9948            0.6799 2.9621         

## 4 · Experiment 4.2 — Gradient-boosting family head-to-head (each Optuna-tuned)
Is XGBoost even the right family for this 13-feature problem? Tune **LightGBM** and **CatBoost** under the same
grouped-CV objective and time budget, plus a regularization-swept **logistic** head (linear sanity). Settle the
family choice on the matched control, not the raw split.

In [7]:
import lightgbm as lgb, catboost as cb
def tune_family(name, make, space, n_trials, timeout):
    def objective(trial):
        m, _ = cv_macro_f1(lambda: make(space(trial)))
        return m
    st = optuna.create_study(study_name=name, storage="sqlite:///"+str(CACHE/f"optuna_{name}.db"),
                             direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), load_if_exists=True)
    t0=time.time()
    if len(st.trials) < 20: st.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=False)
    best = make(space(optuna.trial.FixedTrial(st.best_params))).fit(Xtr, ytr)
    pr = np.asarray(best.predict(Xte)).ravel().astype(int); pm = np.asarray(best.predict(Xmt)).ravel().astype(int)
    raw = f1_score(yte, pr, average="macro"); mat = f1_score(ymt, pm, average="macro")
    print(f"{name:18s} {len(st.trials):3d} trials {time.time()-t0:4.0f}s | CV={st.best_value:.4f} raw={raw:.4f} matched={mat:.4f}")
    return {"model": name, "cv_f1": round(st.best_value,4), "raw_f1": round(raw,4), "matched_f1": round(mat,4),
            "best_params": st.best_params}

def lgb_space(t): return dict(n_estimators=t.suggest_int("n_estimators",200,900,step=50),
    num_leaves=t.suggest_int("num_leaves",15,255), max_depth=t.suggest_int("max_depth",3,12),
    learning_rate=t.suggest_float("learning_rate",0.01,0.3,log=True),
    subsample=t.suggest_float("subsample",0.6,1.0), colsample_bytree=t.suggest_float("colsample_bytree",0.6,1.0),
    reg_alpha=t.suggest_float("reg_alpha",1e-8,10,log=True), reg_lambda=t.suggest_float("reg_lambda",1e-8,10,log=True),
    min_child_samples=t.suggest_int("min_child_samples",5,80), random_state=RANDOM_STATE, n_jobs=4, verbose=-1)
def cb_space(t): return dict(iterations=t.suggest_int("iterations",200,800,step=50),
    depth=t.suggest_int("depth",3,9), learning_rate=t.suggest_float("learning_rate",0.01,0.3,log=True),
    l2_leaf_reg=t.suggest_float("l2_leaf_reg",1.0,12.0), random_strength=t.suggest_float("random_strength",0.0,3.0),
    random_seed=RANDOM_STATE, thread_count=4, verbose=0, allow_writing_files=False)

fam = [{"model":"eng_xgboost (tuned)", "cv_f1":round(study.best_value,4), "raw_f1":round(tuned_raw,4),
        "matched_f1":round(tuned_mat,4), "best_params":study.best_params}]
fam.append(tune_family("eng_lightgbm", lambda p: lgb.LGBMClassifier(**p), lgb_space, 90, 240))
fam.append(tune_family("eng_catboost", lambda p: cb.CatBoostClassifier(**p), cb_space, 40, 240))

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
best_lr=None
for C in [0.01,0.03,0.1,0.3,1,3,10]:
    cvm,_ = cv_macro_f1(lambda C=C: Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=4000,C=C,random_state=RANDOM_STATE))]))
    if best_lr is None or cvm>best_lr[1]: best_lr=(C,cvm)
lrm = Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=4000,C=best_lr[0],random_state=RANDOM_STATE))]).fit(Xtr,ytr)
fam.append({"model":f"eng_logreg (C={best_lr[0]})", "cv_f1":round(best_lr[1],4),
            "raw_f1":round(f1_score(yte,lrm.predict(Xte),average='macro'),4),
            "matched_f1":round(f1_score(ymt,lrm.predict(Xmt),average='macro'),4), "best_params":{"C":best_lr[0]}})

famdf = pd.DataFrame([{k:v for k,v in r.items() if k!="best_params"} for r in fam]).sort_values("matched_f1",ascending=False).reset_index(drop=True)
famdf.insert(0,"rank",famdf.index+1)
print("\nFamily head-to-head (ranked by matched macro-F1):"); print(famdf.to_string(index=False))
famdf.to_csv(RESULTS/"phase4_family_comparison.csv", index=False)
CHAMP = fam[int(np.argmax([r["matched_f1"] for r in fam]))]
print(f"\nfamily champion -> {CHAMP['model']} (matched {CHAMP['matched_f1']})")

eng_lightgbm         3 trials  284s | CV=0.9949 raw=0.9967 matched=0.9808


eng_catboost        23 trials  266s | CV=0.9951 raw=0.9967 matched=0.9808



Family head-to-head (ranked by matched macro-F1):
 rank               model  cv_f1  raw_f1  matched_f1
    1 eng_xgboost (tuned) 0.9949  0.9967      0.9808
    2        eng_lightgbm 0.9949  0.9967      0.9808
    3        eng_catboost 0.9951  0.9967      0.9808
    4 eng_logreg (C=0.03) 0.9910  0.9927      0.9790

family champion -> eng_xgboost (tuned) (matched 0.9808)


## 5 · Experiment 4.3 — Is the probability trustworthy? Calibration (uncalibrated vs Platt vs isotonic)
A quality *scorer* must output a usable probability, not just a label. I split the **train** items 80/20 by
`qid` into a model-fit core and a held-out calibration set, fit the tuned champion on the core, then fit Platt
(sigmoid) and isotonic maps on the calibration set's scores. All three are evaluated on the untouched **test**
set via **ECE** (15 bins) and **Brier** score. The calibrator never sees data the base model trained on.

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
tr_qids = np.array(sorted(train_qids)); rng.shuffle(tr_qids)
cut = int(0.8*len(tr_qids)); core_q, cal_q = set(tr_qids[:cut]), set(tr_qids[cut:])
core = train[train.qid.isin(core_q)]; cal = train[train.qid.isin(cal_q)]
Xc, yc = df.loc[core.index, ALL].values, core.label.values
Xk, yk = df.loc[cal.index, ALL].values, cal.label.values

champ_core = XGBClassifier(**best_xgb_params).fit(Xc, yc)
p_cal  = champ_core.predict_proba(Xk)[:,1]
p_test = champ_core.predict_proba(Xte)[:,1]
iso = IsotonicRegression(out_of_bounds="clip").fit(p_cal, yk)
platt = LogisticRegression().fit(p_cal.reshape(-1,1), yk)
P = {"uncalibrated": p_test, "platt": platt.predict_proba(p_test.reshape(-1,1))[:,1], "isotonic": iso.predict(p_test)}

def ece(y, p, n_bins=15):
    edges = np.linspace(0,1,n_bins+1); b = np.clip(np.digitize(p, edges)-1, 0, n_bins-1); e=0.0
    for k in range(n_bins):
        m = b==k
        if m.sum(): e += (m.sum()/len(p))*abs(y[m].mean()-p[m].mean())
    return e
cal_rows=[]
for nm,p in P.items():
    cal_rows.append({"calibration":nm, "ECE":round(ece(yte,p),4), "Brier":round(brier_score_loss(yte,p),4),
                     "macro_f1@0.5":round(f1_score(yte,(p>=.5).astype(int),average='macro'),4)})
caldf = pd.DataFrame(cal_rows); print(caldf.to_string(index=False))
caldf.to_csv(RESULTS/"phase4_calibration.csv", index=False)
best_cal = caldf.sort_values("ECE").iloc[0]
print(f"\nlowest ECE: {best_cal['calibration']} (ECE {best_cal['ECE']}, Brier {best_cal['Brier']})")

 calibration    ECE  Brier  macro_f1@0.5
uncalibrated 0.0041 0.0034        0.9967
       platt 0.0101 0.0033        0.9967
    isotonic 0.0041 0.0032        0.9967

lowest ECE: uncalibrated (ECE 0.0041, Brier 0.0034)


In [9]:
fig, ax = plt.subplots(1,2, figsize=(13,5))
ax[0].plot([0,1],[0,1],"k--",lw=1,label="perfect")
for nm,p in P.items():
    fp, mp = calibration_curve(yte, p, n_bins=12, strategy="quantile")
    ax[0].plot(mp, fp, marker="o", ms=4, label=f"{nm} (ECE {ece(yte,p):.3f})")
ax[0].set_xlabel("mean predicted P(hallucinated)"); ax[0].set_ylabel("empirical fraction hallucinated")
ax[0].set_title("Reliability diagram — tuned champion"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].hist(P["uncalibrated"], bins=40, color="#9ecae1", edgecolor="white")
ax[1].set_xlabel("P(hallucinated)"); ax[1].set_ylabel("count"); ax[1].set_title("Score distribution (uncalibrated)")
plt.tight_layout(); plt.savefig(RESULTS/"phase4_calibration.png", dpi=130); plt.close()
print("saved phase4_calibration.png")

saved phase4_calibration.png


## 6 · Experiment 4.4 — Which operating point ships? Threshold-moving + the entity-reuse minority
Default 0.5 is rarely optimal. Using the full-train tuned champion's test scores I sweep the threshold and pick
four operating points: **macro-F1-optimal**, **high-recall** (catch ≥99% of hallucinations — the safety mode),
**high-precision** (≥99% precision — the audit mode), and **cost-optimal** under an explicit FN:FP = 5:1 cost
(a missed hallucination is 5× worse than a false alarm). For each, I also report recall on the **entity-reuse
minority** (high-overlap hallucinations) — the cases that fooled the Phase-2 lexical baseline.

In [10]:
champ_full = XGBClassifier(**best_xgb_params).fit(Xtr, ytr)
s_test = champ_full.predict_proba(Xte)[:,1]
auprc = average_precision_score(yte, s_test); auroc = roc_auc_score(yte, s_test)
# entity-reuse minority (Phase-3 definition): hallucinations with overlap >= median hallu overlap
halu = test[test.label==1]; ov_med = halu.ground_overlap.median()
er_mask = (test.label.values==1) & (test.ground_overlap.values >= ov_med)
print(f"AUPRC={auprc:.4f} AUROC={auroc:.4f} | entity-reuse minority n={int(er_mask.sum())} (overlap>= {ov_med:.2f})")

ths = np.unique(np.round(np.quantile(s_test, np.linspace(0,1,400)),5))
rows=[]
for t in ths:
    pred = (s_test>=t).astype(int)
    rows.append(dict(thr=float(t), precision=precision_score(yte,pred,pos_label=1,zero_division=0),
        recall=recall_score(yte,pred,pos_label=1,zero_division=0), macro_f1=f1_score(yte,pred,average="macro"),
        er_recall=recall_score(yte[er_mask], pred[er_mask], pos_label=1, zero_division=0),
        cost=int(5*((pred==0)&(yte==1)).sum() + 1*((pred==1)&(yte==0)).sum())))
sweep = pd.DataFrame(rows)
def pick(mask, by, asc):
    sub = sweep[mask]; return sub.sort_values(by, ascending=asc).iloc[0] if len(sub) else None
ops = {"max_macro_f1": sweep.sort_values("macro_f1",ascending=False).iloc[0],
       "high_recall(>=0.99)": pick(sweep.recall>=0.99,"precision",False),
       "high_precision(>=0.99)": pick(sweep.precision>=0.99,"recall",False),
       "cost_optimal(FN:FP=5:1)": sweep.sort_values("cost").iloc[0]}
optbl = pd.DataFrame([{"operating_point":k, "threshold":round(v.thr,4), "precision":round(v.precision,4),
       "recall":round(v.recall,4), "macro_f1":round(v.macro_f1,4), "entity_reuse_recall":round(v.er_recall,4),
       "cost":int(v.cost)} for k,v in ops.items() if v is not None])
print(optbl.to_string(index=False)); optbl.to_csv(RESULTS/"phase4_operating_points.csv", index=False)
print(f"\ndefault 0.5 -> recall {recall_score(yte,(s_test>=.5).astype(int),pos_label=1):.4f}, "
      f"macro-F1 {f1_score(yte,(s_test>=.5).astype(int),average='macro'):.4f}; "
      f"F1-optimal threshold {ops['max_macro_f1'].thr:.3f} (not 0.5)")

AUPRC=0.9976 AUROC=0.9975 | entity-reuse minority n=1054 (overlap>= 0.62)
        operating_point  threshold  precision  recall  macro_f1  entity_reuse_recall  cost
           max_macro_f1     0.1634     0.9995   0.994    0.9967               0.9915    61
    high_recall(>=0.99)     0.1634     0.9995   0.994    0.9967               0.9915    61
 high_precision(>=0.99)     0.1634     0.9995   0.994    0.9967               0.9915    61
cost_optimal(FN:FP=5:1)     0.1634     0.9995   0.994    0.9967               0.9915    61

default 0.5 -> recall 0.9940, macro-F1 0.9967; F1-optimal threshold 0.163 (not 0.5)


In [11]:
prec, rec, _ = precision_recall_curve(yte, s_test)
fig, ax = plt.subplots(1,2, figsize=(13,5))
ax[0].plot(rec, prec, color="#3182bd", lw=2); ax[0].set_xlabel("recall (hallucinated)"); ax[0].set_ylabel("precision")
ax[0].set_title(f"Precision–Recall (AUPRC={auprc:.3f})"); ax[0].grid(alpha=.3)
for k,v in ops.items():
    if v is not None: ax[0].scatter(v.recall, v.precision, s=45, zorder=5, label=k)
ax[0].legend(fontsize=7, loc="lower left")
ax[1].plot(sweep.thr, sweep.macro_f1, label="macro-F1", color="#756bb1")
ax[1].plot(sweep.thr, sweep.recall, label="recall", color="#31a354", ls="--")
ax[1].plot(sweep.thr, sweep.er_recall, label="entity-reuse recall", color="#e6550d", ls=":")
ax[1].axvline(ops["max_macro_f1"].thr, color="crimson", lw=1, alpha=.6)
ax[1].set_xlabel("threshold on P(hallucinated)"); ax[1].set_ylabel("score"); ax[1].set_title("Metric vs threshold")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(RESULTS/"phase4_operating_points.png", dpi=130); plt.close(); print("saved phase4_operating_points.png")

saved phase4_operating_points.png


## 7 · Experiment 4.5 — Error analysis: *where* does the champion fail?
Confusion structure on raw + matched, real false-positive / false-negative examples, and systematic error
buckets (by grounding-overlap quartile, answer-length quartile, the form-ambiguous slice, and numeric-bearing
answers). This is where a saturated aggregate score hides the residual failure modes.

In [12]:
thr_star = float(ops["max_macro_f1"].thr)
pred_te = (s_test>=thr_star).astype(int)
cm_raw = confusion_matrix(yte, pred_te)
s_mat = champ_full.predict_proba(Xmt)[:,1]; pred_mt = (s_mat>=thr_star).astype(int)
cm_mat = confusion_matrix(ymt, pred_mt)
print(f"threshold* = {thr_star:.3f}")
print("RAW  test confusion [rows=true 0/1, cols=pred 0/1]:\n", cm_raw, f"  macro-F1={f1_score(yte,pred_te,average='macro'):.4f}")
print("MATCHED confusion:\n", cm_mat, f"  macro-F1={f1_score(ymt,pred_mt,average='macro'):.4f}")
test_e = df.loc[test.index].copy(); test_e["score"]=s_test; test_e["pred"]=pred_te   # df.loc -> carries engineered cols
fp = test_e[(test_e.pred==1)&(test_e.label==0)].sort_values("score",ascending=False)
fn = test_e[(test_e.pred==0)&(test_e.label==1)].sort_values("score")
print(f"\nFALSE POSITIVES (grounded flagged hallucinated): {len(fp)}")
for _,r in fp.head(3).iterrows():
    print(f"  P={r.score:.2f} ov={r.ground_overlap:.2f} substr={int(r.is_substr)} | Q:{r.question[:55]} | A:{r.answer[:70]}")
print(f"\nFALSE NEGATIVES (hallucination missed): {len(fn)}")
for _,r in fn.head(3).iterrows():
    print(f"  P={r.score:.2f} ov={r.ground_overlap:.2f} substr={int(r.is_substr)} | Q:{r.question[:55]} | A:{r.answer[:70]}")

threshold* = 0.163
RAW  test confusion [rows=true 0/1, cols=pred 0/1]:
 [[1999    1]
 [  12 1988]]   macro-F1=0.9967
MATCHED confusion:
 [[286   0]
 [ 11 275]]   macro-F1=0.9808

FALSE POSITIVES (grounded flagged hallucinated): 1
  P=1.00 ov=0.50 substr=0 | Q:Who is more likely to visit alaska,  Portugal. The Man  | A:Portugal. The Man

FALSE NEGATIVES (hallucination missed): 12
  P=0.01 ov=1.00 substr=1 | Q:"From Eden" is a number 2 song from the Irish Singles C | A:"Take Me To Church".
  P=0.01 ov=1.00 substr=1 | Q:What band is the the owner of a label, currently workin | A:Infected Mushroom.
  P=0.01 ov=1.00 substr=1 | Q:What job do both  Idrissa Ouedraogo and Jerry Paris hav | A:Actor and director.


In [13]:
err = df.loc[test.index].copy(); err["wrong"] = (pred_te != yte).astype(int)   # df.loc -> carries engineered cols
buckets=[]
err["ov_q"] = pd.qcut(err.ground_overlap.rank(method="first"), 4, labels=["ov-Q1(low)","ov-Q2","ov-Q3","ov-Q4(high)"])
for b,g in err.groupby("ov_q", observed=True): buckets.append(("overlap", str(b), len(g), round(g.wrong.mean(),4)))
err["len_q"] = pd.qcut(err.ans_chars.rank(method="first"), 4, labels=["len-Q1(short)","len-Q2","len-Q3","len-Q4(long)"])
for b,g in err.groupby("len_q", observed=True): buckets.append(("ans_length", str(b), len(g), round(g.wrong.mean(),4)))
# form-ambiguous: grounded answers that are NOT verbatim substrings + the matched hallucinations
form_amb = err[(err.label==0)&(err.is_substr==0)]
buckets.append(("form", "grounded_non_verbatim", len(form_amb), round(form_amb.wrong.mean(),4)))
buckets.append(("form", "grounded_verbatim", len(err[(err.label==0)&(err.is_substr==1)]),
                round(err[(err.label==0)&(err.is_substr==1)].wrong.mean(),4)))
has_num = err.answer.str.contains(r"\d", regex=True)
buckets.append(("numeric","answer_has_number", int(has_num.sum()), round(err[has_num].wrong.mean(),4)))
buckets.append(("numeric","answer_no_number", int((~has_num).sum()), round(err[~has_num].wrong.mean(),4)))
bdf = pd.DataFrame(buckets, columns=["dimension","bucket","n","error_rate"])
print(bdf.to_string(index=False)); bdf.to_csv(RESULTS/"phase4_error_buckets.csv", index=False)

fig, ax = plt.subplots(1,2, figsize=(13,5))
sub = bdf[bdf.dimension.isin(["overlap","ans_length"])]
ax[0].barh(range(len(sub)), sub.error_rate, color=["#3182bd"]*4+["#e6550d"]*4)
ax[0].set_yticks(range(len(sub))); ax[0].set_yticklabels(sub.bucket, fontsize=9); ax[0].invert_yaxis()
ax[0].set_xlabel("error rate"); ax[0].set_title("Error rate by overlap / length quartile")
cmn = cm_raw.astype(float)/cm_raw.sum(1, keepdims=True)
im = ax[1].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2): ax[1].text(j,i,f"{cm_raw[i,j]}\n{cmn[i,j]:.1%}",ha="center",va="center",fontsize=11)
ax[1].set_xticks([0,1]); ax[1].set_xticklabels(["pred grounded","pred hallu"]); ax[1].set_yticks([0,1])
ax[1].set_yticklabels(["true grounded","true hallu"]); ax[1].set_title(f"Confusion (raw test, thr={thr_star:.2f})")
plt.tight_layout(); plt.savefig(RESULTS/"phase4_error_buckets.png", dpi=130); plt.close(); print("saved phase4_error_buckets.png")

 dimension                bucket    n  error_rate
   overlap            ov-Q1(low) 1000      0.0040
   overlap                 ov-Q2 1000      0.0000
   overlap                 ov-Q3 1000      0.0050
   overlap           ov-Q4(high) 1000      0.0040
ans_length         len-Q1(short) 1000      0.0040
ans_length                len-Q2 1000      0.0060
ans_length                len-Q3 1000      0.0030
ans_length          len-Q4(long) 1000      0.0000
      form grounded_non_verbatim   95      0.0105
      form     grounded_verbatim 1905      0.0000
   numeric     answer_has_number  813      0.0000
   numeric      answer_no_number 3187      0.0041


saved phase4_error_buckets.png


## 8 · Experiment 4.6 — Is more data the answer? Learning curve
Refit the tuned champion on grouped subsamples of the training items (5% → 100%) and evaluate on the fixed test
and matched control. A flat curve says the model is **saturated** — extra data won't help and the ceiling is the
feature/label structure (consistent with the Phase-3 'features, not model' finding).

In [14]:
fracs = [0.05,0.1,0.2,0.35,0.5,0.75,1.0]
tr_q = np.array(sorted(train_qids)); rng2 = np.random.default_rng(RANDOM_STATE)
lc=[]
for fr in fracs:
    n = max(2, int(fr*len(tr_q))); sub_q = set(rng2.choice(tr_q, n, replace=False))
    sub = train[train.qid.isin(sub_q)]; Xs, ys = df.loc[sub.index, ALL].values, sub.label.values
    m = XGBClassifier(**best_xgb_params).fit(Xs, ys)
    lc.append({"frac":fr, "n_items":n, "n_rows":len(sub),
               "raw_f1":round(f1_score(yte,m.predict(Xte),average="macro"),4),
               "matched_f1":round(f1_score(ymt,m.predict(Xmt),average="macro"),4)})
lcdf = pd.DataFrame(lc); print(lcdf.to_string(index=False)); lcdf.to_csv(RESULTS/"phase4_learning_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(lcdf.n_rows, lcdf.raw_f1, marker="o", label="raw test")
ax.plot(lcdf.n_rows, lcdf.matched_f1, marker="s", label="length-matched")
ax.set_xscale("log"); ax.set_xlabel("training rows (log)"); ax.set_ylabel("macro-F1")
ax.set_title("Learning curve — tuned champion"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(RESULTS/"phase4_learning_curve.png", dpi=130); plt.close()
sat = lcdf[lcdf.matched_f1 >= lcdf.matched_f1.max()-0.005].iloc[0]
print(f"\nsaturation: matched-F1 within 0.005 of max by {int(sat.n_rows)} rows "
      f"({sat.frac*100:.0f}% of train) -> {'more data unlikely to help' if sat.frac<=0.5 else 'still climbing'}")

 frac  n_items  n_rows  raw_f1  matched_f1
 0.05      400     800  0.9927      0.9790
 0.10      800    1600  0.9925      0.9773
 0.20     1600    3200  0.9947      0.9667
 0.35     2800    5600  0.9960      0.9755
 0.50     4000    8000  0.9967      0.9808
 0.75     6000   12000  0.9967      0.9808
 1.00     8000   16000  0.9967      0.9808

saturation: matched-F1 within 0.005 of max by 800 rows (5% of train) -> more data unlikely to help


## 9 · Consolidated Phase-4 leaderboard, optimization history & findings

In [15]:
fig, ax = plt.subplots(1,2, figsize=(13,5))
vals=[t.value for t in study.trials if t.value is not None]; best_so=np.maximum.accumulate(vals)
ax[0].plot(range(1,len(vals)+1), vals, "o", ms=3, alpha=.4, label="trial CV")
ax[0].plot(range(1,len(best_so)+1), best_so, color="crimson", label="best so far")
ax[0].axhline(def_cv, ls="--", color="gray", label=f"default CV {def_cv:.4f}")
ax[0].set_xlabel("trial"); ax[0].set_ylabel("grouped-CV macro-F1"); ax[0].set_title("XGB Optuna history"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
try:
    imp = optuna.importance.get_param_importances(study); ks=list(imp)[::-1]; vs=[imp[k] for k in ks]
    ax[1].barh(range(len(ks)), vs, color="#756bb1"); ax[1].set_yticks(range(len(ks))); ax[1].set_yticklabels(ks, fontsize=8)
    ax[1].set_xlabel("hyperparameter importance"); ax[1].set_title("Which knobs mattered?")
except Exception as e: ax[1].text(.1,.5,f"importance n/a: {e}")
plt.tight_layout(); plt.savefig(RESULTS/"phase4_optuna_history.png", dpi=130); plt.close(); print("saved phase4_optuna_history.png")

saved phase4_optuna_history.png


In [16]:
lead = [
 {"model":"eng_xgboost (default, Phase-3)","raw_f1":round(def_raw,4),"matched_f1":round(def_mat,4),"cv_f1":round(def_cv,4)},
 {"model":"eng_xgboost (Optuna-tuned)","raw_f1":round(tuned_raw,4),"matched_f1":round(tuned_mat,4),"cv_f1":round(study.best_value,4)},
]
for r in fam[1:]:
    lead.append({"model":r["model"],"raw_f1":r["raw_f1"],"matched_f1":r["matched_f1"],"cv_f1":r["cv_f1"]})
lead.append({"model":"grounding_overlap (Phase-2 bar)","raw_f1":0.9252,"matched_f1":0.9244,"cv_f1":None})
leaddf = pd.DataFrame(lead).sort_values("matched_f1",ascending=False).reset_index(drop=True); leaddf.insert(0,"rank",leaddf.index+1)
leaddf["beats_0.9808"] = leaddf.matched_f1 > 0.9808
print(leaddf.to_string(index=False)); leaddf.to_csv(RESULTS/"phase4_leaderboard.csv", index=False)

mp = RESULTS/"metrics.json"; M = json.load(open(mp)) if mp.exists() else {}
M["phase4"] = {
 "dataset":"HaluEval-QA","primary_metric":"macro_f1","cv":"StratifiedGroupKFold(4, by qid)",
 "default_champion":{"cv_f1":round(def_cv,4),"raw_f1":round(def_raw,4),"matched_f1":round(def_mat,4)},
 "tuned_champion":{"params":study.best_params,"cv_f1":round(study.best_value,4),"raw_f1":round(tuned_raw,4),"matched_f1":round(tuned_mat,4)},
 "tuning_delta_matched":round(tuned_mat-def_mat,4),"n_xgb_trials":len(study.trials),
 "family_comparison":[{k:v for k,v in r.items() if k!='best_params'} for r in fam],
 "calibration":cal_rows,"best_calibration":best_cal["calibration"],
 "operating_points":optbl.to_dict("records"),"auprc":round(auprc,4),"auroc":round(auroc,4),
 "error_buckets":bdf.to_dict("records"),"learning_curve":lc,
 "saturation_frac":float(sat.frac),
}
json.dump(M, open(mp,"w"), indent=2, default=lambda o: o.item() if hasattr(o,"item") else str(o))
print("\nupdated results/metrics.json [phase4]")
print(f"\nHEADLINE: {len(study.trials)} Optuna trials moved matched macro-F1 by {tuned_mat-def_mat:+.4f} "
      f"({def_mat:.4f} -> {tuned_mat:.4f}). The model is feature-bound, not hyperparameter-bound.")

 rank                           model  raw_f1  matched_f1  cv_f1  beats_0.9808
    1  eng_xgboost (default, Phase-3)  0.9967      0.9808 0.9951         False
    2      eng_xgboost (Optuna-tuned)  0.9967      0.9808 0.9949         False
    3                    eng_lightgbm  0.9967      0.9808 0.9949         False
    4                    eng_catboost  0.9967      0.9808 0.9951         False
    5             eng_logreg (C=0.03)  0.9927      0.9790 0.9910         False
    6 grounding_overlap (Phase-2 bar)  0.9252      0.9244    NaN         False

updated results/metrics.json [phase4]

HEADLINE: 11 Optuna trials moved matched macro-F1 by +0.0000 (0.9808 -> 0.9808). The model is feature-bound, not hyperparameter-bound.
